#### Key Value Cache

In [ ]:
import threading
from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer
from time import time

# Loading the  model and the tokenizer
tokenizer = AutoTokenizer.from_pretrained('gpt2')
model = AutoModelForCausalLM.from_pretrained('gpt2')

prompt = "The next day is bright"
tokens = tokenizer.encode(prompt, return_tensors = "pt")

# function for streaming
def stream_output(use_cache):
    streamer = TextIteratorStreamer(tokenizer, skip_special_tokens = True)

    thread = threading.Thread(target = model.generate, kwargs ={
        "input_ids": tokens,
        "max_new_tokens": 100,
        "use_cache": use_cache,
        "streamer": streamer
    })

    thread.start()

    start_time = time()
    for token in streamer:
        print(token, end = "", flush = True)

    end_time = time()
    thread.join()

    elapsed = end_time - start_time
    print(f"\n\nUse Cache = {use_cache}, Time taken : {elapsed:.3f} seconds\n")

print("==Without KV Caching==")
stream_output(use_cache = False)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


==Without KV Caching==
The next day is bright and sunny, and the sun is shining. The sun is shining, and the moon is shining. The sun is shining, and the moon is shining. The sun is shining, and the moon is shining. The sun is shining, and the moon is shining. The sun is shining, and the moon is shining. The sun is shining, and the moon is shining. The sun is shining, and the moon is shining. The sun is shining, and the moon is shining. The sun is

Use Cache = False, Time taken : 47.489 seconds



#### With and Without KV cache

In [2]:
print("==With KV Caching==")
stream_output(use_cache = True)

print("==Without KV Caching==")
stream_output(use_cache = False)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


==With KV Caching==
The next day is bright and sunny, and the sun is shining. The sun is shining, and the moon is shining. The sun is shining, and the moon is shining. The sun is shining, and the moon is shining. The sun is shining, and the moon is shining. The sun is shining, and the moon is shining. The sun is shining, and the moon is shining. The sun is shining, and the moon is shining. The sun is shining, and the moon is shining. The sun is

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.




Use Cache = True, Time taken : 3.946 seconds

==Without KV Caching==
The next day is bright and sunny, and the sun is shining. The sun is shining, and the moon is shining. The sun is shining, and the moon is shining. The sun is shining, and the moon is shining. The sun is shining, and the moon is shining. The sun is shining, and the moon is shining. The sun is shining, and the moon is shining. The sun is shining, and the moon is shining. The sun is shining, and the moon is shining. The sun is

Use Cache = False, Time taken : 19.097 seconds



#### Comparing them side by side

In [ ]:
import threading
from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer
import time
import os

tokenizer = AutoTokenizer.from_pretrained('gpt2')
model = AutoModelForCausalLM.from_pretrained('gpt2')

prompt = "The next day is bright"
tokens = tokenizer.encode(prompt, return_tensors = "pt")

# streaming function for use in threading
def stream_tokens(use_cache, output_list, timing_list):
    start_time = time.time()
    streamer = TextIteratorStreamer(tokenizer, skip_special_tokens = True)

    generation_kwags = {
        "input_ids": tokens,
        "max_new_tokens": 50,
        "use_cache": use_cache,
        "streamer": streamer,
        "do_sample" : True, # enable sampling
        "temperature": 0.7, # add temperature for more variation
        "top_p": 0.9 # use top_p sampling
    }

    # starting generation in a separate thread
    thread = threading.Thread(target = model.generate, kwags = generation_kwags)
    thread.start()

    # collecting tokens as they are generated
    for token in streamer:
        current_time = time.time() - start_time
        timing_list.append(current_time)
        output_list.append(token)
        time.sleep(0.05) #simualting realistic streaming delay

# Output and timing container
output_with_cache = []
output_without_cache = []
timing_with_cache = []
timing_without_cache = []

# overall start time
overall_start = time.time()

# Starting both generation process
thread_with_cache = threading.Thread(
    target = stream_tokens,
    args=(True, output_with_cache, timing_with_cache)
)

thread_without_cache = threading.Thread(
    target= stream_tokens,
    args=(False, output_without_cache, timing_without_cache)
)

thread_with_cache.start()
thread_without_cache.start()

# Letting the threads run a bit to start generating tokens
time.sleep(1)

# Clear screen
os.system('cls' if os.name == 'nt' else 'clear')
print("=" * 70)
print(f"{'KV CACHE STREAMING COMPARISON(SIDE BY SIDE)'}:^70")
print("=" * 70)
print(f"{'WITH KV CACHE':<33} | {'WITHOUT KV CACHE': <33}")
print("=" * 70)

# Displaying the tokens side by side as they are generated
while (thread_with_cache.is_alive() or thread_without_cache.is_alive() or 
       len(output_with_cache) >0 or len(output_without_cache) >0 ):
    
    # collecting a token from each output
    token_with = output_with_cache.pop(0) if output_without_cache else ""
    token_without = output_without_cache.pop(0) if output_with_cache else ""
     
    if token_with or token_without: # only prints if at least one has a token
        print(f"{token_with: <33}")
        time.sleep(0.05) 
    else:
        # If both are empty but threads are running wait a bit
        if thread_with_cache.is_alive() or thread_without_cache.is_alive():
            time.sleep(0.1)
        else:
            # If no tokens and threads are done, exit the loop
            break

# Wait for threads to complete
thread_with_cache.join()
thread_without_cache.join()

total_time = time.time() - overall_start

# Compiling the full text
with_cache_text = prompt + " " + " ".join(output_with_cache)
without_cache_text = prompt + " " + " ".join(output_without_cache)

print("=" * 70)
print(f"\nStreaming comparison complete in {total_time:.2f} seconds")

# Printing time stats
print("\n== TIMING COMPARISON")
print(f"Total tokens generated:")
print(f"- With KV Cache: {len(timing_with_cache)} tokens in {timing_with_cache[-1]:.2f}s")
print(f"- Without KV Cache: {len(timing_without_cache)} tokens in {timing_without_cache[-1]:.2f}s")

# Calcualting average token speed
if timing_with_cache:
    with_cache_avg = timing_with_cache[-1] / len(timing_with_cache)
    print(f"- With KV Cache avg time per token: {with_cache_avg:.4f}s")

if timing_without_cache:
    without_cache_avg = timing_without_cache[-1] / len(timing_without_cache)
    print(f"- Without KV Cache avg time per token: {without_cache_avg:.4f}s")

if timing_with_cache and timing_without_cache:
    speedup = without_cache_avg / with_cache_avg if with_cache_avg >0 else 0
    print(f"- KV Cache speedup: {speedup:.2f}x faster")

# Printing the generated text
print("\n==Generated Text==")
print(f"\nWith KV Cache:\n{with_cache_text}")
print(f"\nWithout KV Cache:\n{without_cache_text}")


Exception in thread Thread-500 (stream_tokens):
Traceback (most recent call last):
  File "/usr/local/python/3.12.1/lib/python3.12/threading.py", line 1073, in _bootstrap_inner
Exception in thread Thread-501 (stream_tokens):
Traceback (most recent call last):
  File "/usr/local/python/3.12.1/lib/python3.12/threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "/usr/local/python/3.12.1/lib/python3.12/site-packages/ipykernel/ipkernel.py", line 766, in run_closure
    self.run()
  File "/usr/local/python/3.12.1/lib/python3.12/site-packages/ipykernel/ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "/usr/local/python/3.12.1/lib/python3.12/threading.py", line 1010, in run
    _threading_Thread_run(self)
  File "/usr/local/python/3.12.1/lib/python3.12/threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_1644/1045252490.py", line 28, in stream_tokens
    self._target(*self._args, **self._kwargs)
  File

KV CACHE STREAMING COMPARISON(SIDE BY SIDE):^70
WITH KV CACHE                     | WITHOUT KV CACHE                 

Streaming comparison complete in 1.00 seconds

== TIMING COMPARISON
Total tokens generated:


IndexError: list index out of range

In [ ]:
import torch
import time
import threading
from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained('gpt2')
model = AutoModelForCausalLM.from_pretrained('gpt2').to(device)

prompt = "The next day is bright"
tokens = tokenizer.encode(prompt, return_tensors = "pt").to(device)

# streaming function for use in threading
def stream_tokens(use_cache, output_list, timing_list):
    start_time = time.time()
    streamer = TextIteratorStreamer(tokenizer, skip_special_tokens = True)

    generation_kwags = {
        "input_ids": tokens,
        "max_new_tokens": 50,
        "use_cache": use_cache,
        "streamer": streamer,
        "do_sample" : True, # enable sampling
        "temperature": 0.7, # add temperature for more variation
        "top_p": 0.9 # use top_p sampling
    }

    # starting generation in a separate thread
    thread = threading.Thread(target = model.generate, kwags = generation_kwags)
    thread.start()

    # collecting tokens as they are generated
    for token in streamer:
        current_time = time.time() - start_time
        timing_list.append(current_time)
        output_list.append(token)
        time.sleep(0.05) #simualting realistic streaming delay

# Output and timing container
output_with_cache = []
output_without_cache = []
timing_with_cache = []
timing_without_cache = []

# overall start time
overall_start = time.time()

# Starting both generation process
thread_with_cache = threading.Thread(
    target = stream_tokens,
    args=(True, output_with_cache, timing_with_cache)
)

thread_without_cache = threading.Thread(
    target= stream_tokens,
    args=(False, output_without_cache, timing_without_cache)
)

thread_with_cache.start()
thread_without_cache.start()

# Letting the threads run a bit to start generating tokens
time.sleep(1)

# Clear screen
os.system('cls' if os.name == 'nt' else 'clear')
print("=" * 70)
print(f"{'KV CACHE STREAMING COMPARISON(SIDE BY SIDE)'}:^70")
print("=" * 70)
print(f"{'WITH KV CACHE':<33} | {'WITHOUT KV CACHE': <33}")
print("=" * 70)

# Displaying the tokens side by side as they are generated
while (thread_with_cache.is_alive() or thread_without_cache.is_alive() or 
       len(output_with_cache) >0 or len(output_without_cache) >0 ):
    
    # collecting a token from each output
    token_with = output_with_cache.pop(0) if output_without_cache else ""
    token_without = output_without_cache.pop(0) if output_with_cache else ""
     
    if token_with or token_without: # only prints if at least one has a token
        print(f"{token_with: <33}")
        time.sleep(0.05) 
    else:
        # If both are empty but threads are running wait a bit
        if thread_with_cache.is_alive() or thread_without_cache.is_alive():
            time.sleep(0.1)
        else:
            # If no tokens and threads are done, exit the loop
            break

# Wait for threads to complete
thread_with_cache.join()
thread_without_cache.join()

total_time = time.time() - overall_start

# Compiling the full text
with_cache_text = prompt + " " + " ".join(output_with_cache)
without_cache_text = prompt + " " + " ".join(output_without_cache)

print("=" * 70)
print(f"\nStreaming comparison complete in {total_time:.2f} seconds")

# Printing time stats
print("\n== TIMING COMPARISON")
print(f"Total tokens generated:")
print(f"- With KV Cache: {len(timing_with_cache)} tokens in {timing_with_cache[-1]:.2f}s")
print(f"- Without KV Cache: {len(timing_without_cache)} tokens in {timing_without_cache[-1]:.2f}s")

# Calcualting average token speed
if timing_with_cache:
    with_cache_avg = timing_with_cache[-1] / len(timing_with_cache)
    print(f"- With KV Cache avg time per token: {with_cache_avg:.4f}s")

if timing_without_cache:
    without_cache_avg = timing_without_cache[-1] / len(timing_without_cache)
    print(f"- Without KV Cache avg time per token: {without_cache_avg:.4f}s")

if timing_with_cache and timing_without_cache:
    speedup = without_cache_avg / with_cache_avg if with_cache_avg >0 else 0
    print(f"- KV Cache speedup: {speedup:.2f}x faster")

# Printing the generated text
print("\n==Generated Text==")
print(f"\nWith KV Cache:\n{with_cache_text}")
print(f"\nWithout KV Cache:\n{without_cache_text}")


In [13]:
import torch
import time
import threading
from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer

# --- Setup Model and Tokenizer ---
# Use a GPU if available, as the performance difference is more dramatic.
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained('gpt2')
model = AutoModelForCausalLM.from_pretrained('gpt2').to(device)

prompt = "The next day is bright"
tokens = tokenizer.encode(prompt, return_tensors="pt").to(device)

# --- Define a Reusable Generation Function ---
# This function runs one generation and returns the generated text and total time.
def generate_and_time(use_cache_flag):
    """
    Generates text from the model and times the process.
    - use_cache_flag (bool): Whether to use the KV cache in the model.
    """
    streamer = TextIteratorStreamer(tokenizer, skip_special_tokens=True)

    generation_kwargs = {
        "input_ids": tokens,
        "max_new_tokens": 100, # Increased token count to better see the speed difference
        "use_cache": use_cache_flag,
        "streamer": streamer,
        "do_sample": True,
        "temperature": 0.7,
        "top_p": 0.9,
    }

    # model.generate must run in a thread for the streamer to work correctly.
    # This is safe because we only run one generation at a time.
    thread = threading.Thread(target=model.generate, kwargs=generation_kwargs)
    
    print("=" * 70)
    print(f"STARTING GENERATION (use_cache = {use_cache_flag})")
    print("=" * 70)
    print(prompt, end="", flush=True)

    start_time = time.time()
    thread.start()

    # Collect the generated text as it streams
    generated_text = ""
    for new_text in streamer:
        print(new_text, end="", flush=True)
        generated_text += new_text
    
    # Wait for the thread to finish
    thread.join()
    end_time = time.time()
    
    total_time = end_time - start_time
    print(f"\n\n--- Generation finished in {total_time:.2f} seconds ---\n")
    
    return generated_text, total_time


# --- Main Execution ---

# 1. Run with KV Cache enabled
text_with_cache, time_with_cache = generate_and_time(use_cache_flag=True)

# 2. Run with KV Cache disabled
text_without_cache, time_without_cache = generate_and_time(use_cache_flag=False)


# --- Final Comparison ---
print("=" * 70)
print(f"{'KV CACHE BENCHMARK COMPLETE':^70}")
print("=" * 70)

# Calculate token counts for accurate "per token" metrics
tokens_with_cache = tokenizer.encode(text_with_cache)
tokens_without_cache = tokenizer.encode(text_without_cache)

num_tokens_with_cache = len(tokens_with_cache)
num_tokens_without_cache = len(tokens_without_cache)

if num_tokens_with_cache > 0:
    avg_with_cache = time_with_cache / num_tokens_with_cache
    print(f"With KV Cache:")
    print(f"  - Generated {num_tokens_with_cache} tokens in {time_with_cache:.2f} seconds.")
    print(f"  - Average time per token: {avg_with_cache:.4f}s")
    print(f"  - Full Text: {prompt}{text_with_cache}\n")

if num_tokens_without_cache > 0:
    avg_without_cache = time_without_cache / num_tokens_without_cache
    print(f"Without KV Cache:")
    print(f"  - Generated {num_tokens_without_cache} tokens in {time_without_cache:.2f} seconds.")
    print(f"  - Average time per token: {avg_without_cache:.4f}s")
    print(f"  - Full Text: {prompt}{text_without_cache}\n")

# Calculate and print the final speedup factor
if num_tokens_with_cache > 0 and num_tokens_without_cache > 0 and avg_with_cache > 0:
    speedup = avg_without_cache / avg_with_cache
    print("-" * 70)
    print(f"✅ KV Cache Speedup: {speedup:.2f}x faster per token.")
    print("-" * 70)

Using device: cpu
STARTING GENERATION (use_cache = True)
The next day is bright

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


The next day is bright, but the only thing that is bright is the sun. It's a big day.

"I was just thinking about how much I can make of the weather, how much I can make of the energy I have, how much I can make of the time I spend in the sun, how much I can make of the time I live in the darkness," he said.

"I think we are going to see some very different things going on with the sun. We have to

--- Generation finished in 4.87 seconds ---

STARTING GENERATION (use_cache = False)
The next day is bright

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


The next day is bright and sunny, but we can't see it."

When asked about the possibility of a possible invasion, she said: "I have no idea."

The BBC understands that the BBC is planning to launch a campaign on the matter, which it says will include a special programme on the possibility of a "jihad".

A spokesperson for the BBC said: "We are confident that we have an air-to-air relationship with the BBC and have taken all appropriate steps to ensure

--- Generation finished in 17.61 seconds ---

                     KV CACHE BENCHMARK COMPLETE                      
With KV Cache:
  - Generated 105 tokens in 4.87 seconds.
  - Average time per token: 0.0464s
  - Full Text: The next day is brightThe next day is bright, but the only thing that is bright is the sun. It's a big day.

"I was just thinking about how much I can make of the weather, how much I can make of the energy I have, how much I can make of the time I spend in the sun, how much I can make of the time I live in the darkne

In [15]:
import torch
import time
import threading
from queue import Queue, Empty
from transformers import AutoTokenizer, AutoModelForCausalLM

# --- Setup ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained('gpt2')
prompt = "The next day is bright"
tokens = tokenizer.encode(prompt, return_tensors="pt").to(device)

# --- IMPORTANT: Load two separate model instances ---
print("Loading model for 'With Cache' stream...")
model_with_cache = AutoModelForCausalLM.from_pretrained('gpt2').to(device)

print("Loading model for 'Without Cache' stream...")
model_without_cache = AutoModelForCausalLM.from_pretrained('gpt2').to(device)

# --- Generator function to be run in a thread ---
def stream_generator(model, use_cache_flag, token_queue, results_dict):
    """
    Runs model.generate, puts tokens into a queue, and records the total time.
    """
    from transformers import TextStreamer

    class QueueStreamer(TextStreamer):
        def __init__(self, queue, tokenizer):
            super().__init__(tokenizer)
            self.queue = queue
        def on_finalized_text(self, text: str, stream_end: bool = False):
            self.queue.put(text)
            if stream_end:
                self.queue.put(None) # Sentinel to signal the end

    streamer = QueueStreamer(token_queue, tokenizer)
    generation_kwargs = {
        "input_ids": tokens, "max_new_tokens": 50, "use_cache": use_cache_flag,
        "streamer": streamer, "do_sample": True, "temperature": 0.7, "top_p": 0.9,
    }

    start_time = time.time()
    model.generate(**generation_kwargs)
    end_time = time.time()
    results_dict['time'] = end_time - start_time # Store time in the shared dict

# --- Main Execution ---

q_with_cache = Queue()
q_without_cache = Queue()
results_w = {}
results_wo = {}

thread_with_cache = threading.Thread(target=stream_generator, args=(model_with_cache, True, q_with_cache, results_w))
thread_without_cache = threading.Thread(target=stream_generator, args=(model_without_cache, False, q_without_cache, results_wo))

thread_with_cache.start()
thread_without_cache.start()

# --- Corrected Display Loop ---
print("\n" + "=" * 80)
print(f"{'KV CACHE STREAMING COMPARISON (Side-by-side)':^80}")
print("=" * 80)
print(f"{'WITH KV CACHE':<39} | {'WITHOUT KV CACHE'}")
print("-" * 80)
print(f"{prompt:<39} | {prompt}")

text_w, text_wo = [], []
w_done, wo_done = False, False

while not w_done or not wo_done:
    new_w, new_wo = "", ""
    try:
        # Get all available tokens from the queue for this iteration
        while (token := q_with_cache.get(block=False)) is not None:
            new_w += token
            text_w.append(token)
        w_done = True
    except Empty:
        pass
    try:
        while (token := q_without_cache.get(block=False)) is not None:
            new_wo += token
            text_wo.append(token)
        wo_done = True
    except Empty:
        pass

    # Print only if there's new text for either column
    if new_w or new_wo:
        # We use carriage return '\r' to redraw the line, which can be complex.
        # A simpler way is to just print new lines.
        print(f"{new_w:<39} | {new_wo}")

    time.sleep(0.05)

thread_with_cache.join()
thread_without_cache.join()

print("\n" + "=" * 80)
print(f"{'BENCHMARK COMPLETE':^80}")
print("=" * 80)

# --- Final Timing Statistics ---
time_with_cache = results_w.get('time', 0)
time_without_cache = results_wo.get('time', 0)

full_text_w = "".join(text_w)
full_text_wo = "".join(text_wo)
num_tokens_w = len(tokenizer.encode(full_text_w))
num_tokens_wo = len(tokenizer.encode(full_text_wo))

if num_tokens_w > 0:
    avg_w = time_with_cache / num_tokens_w
    print("With KV Cache:")
    print(f"  - Generated {num_tokens_w} tokens in {time_with_cache:.2f} seconds.")
    print(f"  - Average tokens per second: {num_tokens_w / time_with_cache:.2f}")

if num_tokens_wo > 0:
    avg_wo = time_without_cache / num_tokens_wo
    print("\nWithout KV Cache:")
    print(f"  - Generated {num_tokens_wo} tokens in {time_without_cache:.2f} seconds.")
    print(f"  - Average tokens per second: {num_tokens_wo / time_without_cache:.2f}")

if num_tokens_w > 0 and num_tokens_wo > 0 and time_with_cache > 0:
    speedup = time_without_cache / time_with_cache
    print("\n" + "-" * 80)
    print(f"✅ Overall Speedup: {speedup:.2f}x faster with KV Cache.")
    print("-" * 80)

Using device: cpu
Loading model for 'With Cache' stream...


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Loading model for 'Without Cache' stream...

                  KV CACHE STREAMING COMPARISON (Side-by-side)                  
WITH KV CACHE                           | WITHOUT KV CACHE
--------------------------------------------------------------------------------
The next day is bright                  | The next day is bright
The next day is                         | The next day is 
bright                                  | 
and                                     | 
                                        | bright, 
sunny,                                  | 
and                                     | and 
I'm                                     | 
                                        | I 
sitting                                 | 
in                                      | have 
the                                     | 
back                                    | 
seat                                    | 
of                                      | 
                                